## Libraries & Function

In [ ]:
import numpy as np
from scipy.linalg import expm
from qutip import *
import numba
from numba import njit, prange
import pickle
import os

In [ ]:
# ==============
# Pauli Matrices
# ==============
sz = np.array(([[1,0], [0,-1]]), dtype=complex); sx = np.array(([[0,1],[1,0]]), dtype=complex); sy = np.array(([[0,-1j],[1j,0]]), dtype=complex) 


In [ ]:
#funzione per plottare in LaTex delle matrici
def array_to_latex(array, real = False, array_name = None):
    array = array.real if real else array
    matrix = ''
    for row in array:
        try:
            for number in row:
                matrix += f'{number}&'
        except TypeError:
            matrix += f'{row}&'
        matrix = matrix[:-1] + r'\\'
    if array_name != None:
        display(Math(array_name+r' = \begin{bmatrix}'+matrix+r'\end{bmatrix}'))
    else:
        display(Math(r'\begin{bmatrix}'+matrix+r'\end{bmatrix}'))

### Hamiltonians and U operator

In [ ]:
def system_Hamiltonian(N_site, E, V_array, mode="complete"):
    """
    Build up of the System's Hamiltonian for the complete basis (ground & excited states) or only excited states.
    
    Method: - "complete"-> complete basis (ground & excited states)
            - "exc"-> excited basis (only excited states)
    
    Parameters: - E: Float, System's Site Energies (randomly generated)
                - V_array: Float, Hopping Potential
                - N_site : Int, Number of Sites
        
    Returns : System's Hamiltonian as Numpy array
    """
    # -----------------------------------------------------
    # Build symmetric matrix from upper triangular elements
    # -----------------------------------------------------
    V_matrix = np.zeros((N_site, N_site))
    idx = 0  # runs over V_array
    for i in range(N_site):
        for j in range(i+1, N_site):
            V_matrix[i, j] = V_array[idx]
            V_matrix[j, i] = V_array[idx]  # Symmetric
            idx += 1
    
    # -------------------------
    # Only Excited States Basis
    # -------------------------
    if mode == "exc":   
        H_sys = np.zeros((N_site, N_site), dtype=complex)
        for i in range(N_site):
            H_sys[i, i] = E[i]
 
        for i in range(N_site):
            for j in range(N_site):
                if i != j:
                    H_sys[i, j] = V_matrix[i, j]
        return H_sys
        
    # --------------
    # Complete Basis 
    # --------------    
    elif mode == "complete":   
        H_sys = np.zeros((2**N_site, 2**N_site), dtype='complex')
        
        for i in range(N_site):
            H_i = (E[i]/2) * (tensor(identity(2**i), identity(2)-sigmaz(), identity(2**(N_site-i-1))))
            H_sys += H_i.full()
            
            for j in range(i+1, N_site):
               H_ij = V_matrix[i, j]/2 * (tensor(identity(2**i), sigmax(), identity(2**(j-i-1)), sigmax(), identity(2**(N_site-j-1))) + tensor(identity(2**i), sigmay(), identity(2**(j-i-1)), sigmay(), identity(2**(N_site-j-1))))
               H_sys += H_ij.full()
        
        return H_sys

    else:
        raise ValueError("mode : 'complete' or 'exc'")


In [ ]:
def interaction_Hamiltonian(N_site, c_CM, method=None):   
    """
    Build up of the Hamiltonian of Interaction for the Collision System - Ancilla in both Quantum Jump and Diffusive Limit
       
    Parameters: - N_site : int, Number of Sites
                - c_CM : list, Interaction Forces for the System - Ancilla intercation/collsion

    Method: - QJ : Quantum Jump Limit
            - Diff : Diffusive Limit
        
    Returns : Hamiltonian of Interaction as Qutip object
    """
    if method is None:
        method = INTERACTION_LIMIT
        
    dim_tot = 2**(2 * N_site)  # total Hilbert Space System + Ancilla
    H_int = np.zeros((dim_tot, dim_tot), dtype=complex)   #inizialization

    # Selection of the Ancilla's operator
    if method == 'QJ':
        anc_op = sigmax() # Interaction Z (sys) - X (anc) -> gives jumps/flip
    elif method == 'Diff':
        anc_op = sigmaz() # Interaction Z (sys) - Z (anc) -> gives dephasing
    else:
        raise ValueError("Method has to be 'QJ' or 'Diff'")
        
    for j in range(N_site):
  
        op_list = [identity(2) for _ in range(2 * N_site)]  #list of identity to be fill with the operator sigmaz & sigmax; 2N identity, N for the system and N fo the ancillas
            
        op_list[j] = sigmaz()      # Acts on the j site
        op_list[N_site + j] = anc_op  # Acts on the j ancilla, with index N + j
            
        H_term = c_CM[j] * tensor(op_list)  # tensor product between the element of the list
            
        H_int += H_term.full()
    
    return H_int


In [ ]:
def hamiltonian_N_ancillas(N_site, E, V_array, c_CM):
    """
    Generation of 3 Hamiltonians for the collision model with N ancillas:
                - H_system : system Hamiltonian
                - H_collision : interaction Hamiltonian with N ancillas
                - H_tot : complete Hamiltonian (system + collision)

    Parameters: - E: Float, System's Site Energies (randomly generated)
                - V_array: Float, Hopping Potential
                - N_site : int, Number of Sites
                - c_CM : list, Interaction Forces for the System - Ancilla intercation/collsion
    
    Returns : H_system, H_collision, H_tot
    """
    
    H_collision = interaction_Hamiltonian(N_site, c_CM)
    
    H_system = system_Hamiltonian(N_site, E, V_array, mode="complete")
        
    dim_anc = 2**N_site
    Id_ancillas = np.eye(dim_anc, dtype=complex)
    H_system_expanded = np.kron(H_system, Id_ancillas)  #expand H_sys in the total space
        
    H_tot = H_system_expanded + H_collision
        
    return H_system, H_collision, H_tot


In [ ]:
def evolution_operator(H, dt, method='expm', hermitian=True):
    """
    Build up of the evolution operator U = exp(-i H dt) using Expm or analytic diagonalization.
   
    Parameters: - H : Qobj or nparray, System Hamiltonian
                - dt : float, Timestep
    
    Method : - "expm"-> build up of the Matrix Exponential with expm
             - "diagonalization"->  build up of the propagater U as V @(exp(-i W dt))@ V_dag with W eigenvalues and V eigenvector of the Hamiltonian 

    Returns : Evolution Operator U, eigenvalues w and eigenvectors V
    """
    H = H.full() if hasattr(H, "full") else np.array(H)
    
    # -----------
    # Expm method
    # -----------
    
    if method == 'expm':
        U = expm(-1j * H * dt)
        return U
        
    # ---------------
    # Diagonalization
    # ---------------
    
    elif method == 'diagonalization':
        if hermitian:
            w, V = np.linalg.eigh(H)
            V_inv = V.conj().T
        else:
            w, V = np.linalg.eig(H) 
            V_inv = np.linalg.inv(V)
                
        U_diag = np.diag(np.exp(-1j * w * dt))
        U = V @ U_diag @ V_inv
        return U, U_diag, w, V

    else:
        raise ValueError("method must be 'expm' or 'diagonalization'")


### Lindblad functions

In [ ]:
def Liouvillian(H, gamma_k, L_k):
    """
    Build the Liouvillian superoperator.
    
    Parameters: - H : nparray, Hamiltonian matrix
                - gamma_k : list, Decay rates
                - L_k : list, Jump Operators
    
    Returns: - super_L : nparray, Liouvillian superoperator
    """    
    I = np.eye(H.shape[0])
    super_L = -1.j * (np.kron(I, H) - np.kron(H.T, I))
    
    for k in range(len(gamma_k)):
        super_L += gamma_k[k] * (np.kron(np.conj(L_k[k]), L_k[k]) - 0.5 * np.kron(I, np.conj(L_k[k]).T @ L_k[k]) - 0.5 * np.kron((np.conj(L_k[k]).T @ L_k[k]).T, I))
    
    return super_L


In [ ]:
@njit(cache=True)
def _evolve_expm_core(super_U, rho_vec_initial, n_times):
    """
    Core evolution loop with expm method (Numba JIT)
    """
    rho_size = rho_vec_initial.shape[0]
    rho_vec_list = np.zeros((rho_size, n_times), dtype=np.complex128)
    rho_vec_list[:, 0] = rho_vec_initial
    
    for i in range(1, n_times):
        rho_vec_list[:, i] = super_U @ rho_vec_list[:, i - 1]
    
    return rho_vec_list


@njit(cache=True)
def _evolve_diagonal_core(V, V_inv, U_diag, rho_vec_initial, n_times):
    """
    Core evolution loop with diagonal method (Numba JIT)
    """
    n_states = len(U_diag)
    
    # Initial coefficients in eigenbasis
    coeff = V_inv @ rho_vec_initial
    coeff_list = np.zeros((n_states, n_times), dtype=np.complex128)
    coeff_list[:, 0] = coeff
    
    # Evolution of coefficients
    for i in range(1, n_times):
        coeff_list[:, i] = U_diag * coeff_list[:, i - 1]
    
    # Transform back to original basis
    rho_vec_list = V @ coeff_list
    
    return rho_vec_list

def Lindblad_evo(rho, H, gamma_k, L_k, times, method="expm", vectorized=True):
    """
    Evolution of the density matrix with the Lindblad Eq. (Optimized with Numba)
    
    Method: - "expm" -> propagator = expm(super_L * dt)
            - "diagonal" -> diagonalization of the super-operator
        
    Vectorized: True/False to choose the output format
    
    Parameters: - H : nparray, System Hamiltonian
                - rho : Qobj or nparray, Initial Density Matrix
                - gamma_k : list, List of Decay Rates
                - L_k : list, List of Jump Operators
                - times : array, Time array
        
    Returns : - if vectorized=True → array (N^2, Nt)
              - if vectorized=False → array (Nt, N_site, N_site)
              - if method="diagonal" also returns V, W
    """
    # Convert to NumPy
    L_k = [L.full() if hasattr(L, "full") else np.array(L, dtype=complex) for L in L_k]
    H = H.full() if hasattr(H, "full") else np.array(H, dtype=complex)
    rho = rho.full() if hasattr(rho, "full") else np.array(rho, dtype=complex)
    
    rho_shape = H.shape[0]
    dt = times[1] - times[0]
    
    # Build Liouvillian
    super_L = Liouvillian(H, gamma_k, L_k)
    
    # Vectorize initial state
    rho_vec = rho.reshape(rho_shape * rho_shape)
    
    # -------------
    # Expm method
    # -------------
    if method == "expm":
        # Compute propagator 
        super_U = expm(super_L * dt)
        
        # evolution loop
        rho_vec_list = _evolve_expm_core(super_U, rho_vec, len(times))
        
        # Output
        if vectorized:
            return rho_vec_list
        else:
            return rho_vec_list.T.reshape(len(times), rho_shape, rho_shape)
    
    # ------------------
    # Diagonal method
    # ------------------
    elif method == "diagonal":
        # Diagonalize Liouvillian 
        W, V = np.linalg.eig(super_L)
        V_inv = np.linalg.inv(V)
        
        # Diagonal evolution operator
        U_diag = np.exp(W * dt)
        
        # evolution loop
        rho_vec_list = _evolve_diagonal_core(V, V_inv, U_diag, rho_vec, len(times))
        
        # Output
        if vectorized:
            return rho_vec_list, V, W
        else:
            return rho_vec_list.T.reshape(len(times), rho_shape, rho_shape), V, W
    
    else:
        raise ValueError("method must be 'expm' or 'diagonal'")

### Isolated System

In [ ]:
@njit(cache=True)
def _compute_trajectory_isolated_core_general(psi_initial, U_site, projectors, n_times):
    """
    Core evolution loop - general for any number of sites
    """
    N_site = projectors.shape[0]
    pop_traj = np.zeros((N_site, n_times), dtype=np.float64)
    
    # Initial populations
    for site in range(N_site):
        pop_traj[site, 0] = np.real(np.vdot(psi_initial, projectors[site] @ psi_initial))
    
    # Evolution
    psi = psi_initial.copy()
    for step in range(1, n_times):
        psi = U_site @ psi
        
        for site in range(N_site):
            pop_traj[site, step] = np.real(np.vdot(psi, projectors[site] @ psi))
    
    return pop_traj

def compute_trajectory_wf_isolated(N_site, times, projectors, psi_sys_initial, U_site):
    """
    Optimized isolated system evolution with Numba (works for any N_site).
    """
    # Convert to NumPy
    U_site_np = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=complex)
    psi_initial_np = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=complex)
    
    # Flatten if needed
    if psi_initial_np.ndim > 1:
        psi_initial_np = psi_initial_np.flatten()
        
    # JIT-compiled evolution
    pop_traj_isolated = _compute_trajectory_isolated_core_general(
        psi_initial_np, U_site_np, projectors, len(times))
    
    return pop_traj_isolated


### Collisional Method functions

#### Evolution with $ U_{complete} $ and Trace on the ancilla

In [ ]:
@njit(cache=True)
def _compute_trace_ancilla_core_general(rho_sys, rho_anc, U_step, U_step_dag, projectors, times, dim_sys, dim_anc, N_site):
    """
    Core computation optimized with Numba - generalized for N sites
    """
    pops_complete = np.zeros((N_site, len(times)), dtype=np.float64)
    
    # -------------
    # initial state
    # -------------
    for site in range(N_site):
        pops_complete[site, 0] = np.real(np.trace(projectors[site] @ rho_sys))
    
    # --------------
    # Time Evolution
    # --------------
    for t in range(1, len(times)):
        # 1: Expansion
        rho_tot = np.kron(rho_sys, rho_anc)
        
        # 2: Evolution
        rho_tot = U_step @ rho_tot @ U_step_dag
        
        # 3: Partial Trace
        rho_tot_reshaped = rho_tot.reshape(dim_sys, dim_anc, dim_sys, dim_anc)
        
        # Manual trace
        rho_sys = np.zeros((dim_sys, dim_sys), dtype=np.complex128)
        for i in range(dim_sys):
            for j in range(dim_sys):
                for k in range(dim_anc):
                    rho_sys[i, j] += rho_tot_reshaped[i, k, j, k]
        
        # 4: Store populations - all sites
        for site in range(N_site):
            pops_complete[site, t] = np.real(np.trace(projectors[site] @ rho_sys))
    
    return pops_complete


def compute_trace_ancilla(rho_sys_initial, U_diag, V, times, projectors, N_site, method=None):
    """
    Compute time evolution with Trace over Ancilla and Reset, using Density Matrix formalism (equal to Infinite Trajectories)

    Parameters: - rho_sys_initial : Np array, Initial Density Matrix
                - U_diag: Np array, Time Evolution Operator for the complete system (S + A) in diagonal form
                - U_diag_dag: Np array, Adjoint operator of U_diag
                - V_diag: Np array, Hopping Potential
                - V_diag_dag: Np array, Adjoint operator of V_diag
                - times : array, Time array
                - N_site : int, Number of Sites                
                - projectors: Np array, Projection Operators on |10> & |01>

    Method: - QJ : Quantum Jump Limit, uses |0><0|
            - Diff : Diffusive Limit, uses I/2
        
    Returns: - pops_complete : np array (N_site x len(times)),  Population evolution for each site
    """
    if method is None:
        method = INTERACTION_LIMIT

    # ------------------------
    # Ancilla's Density Matrix
    # -----------------------
    
    if method == 'QJ':
        
        rho_anc_single = ket2dm(basis(2,0)) # Pure state |0><0|: [[1, 0], [0, 0]]
    
    elif method == 'Diff':
        
        rho_anc_single = (qeye(2) / 2) # Completely Mixed state I/2: [[0.5, 0], [0, 0.5]]
    else:
        raise ValueError("Method must be 'QJ' or 'Diff'")

    rho_anc = (tensor([rho_anc_single for _ in range(N_site)])).full() #for N ancilla
    
    # ----------------
    # Convert to numpy
    # ----------------
    rho_sys = rho_sys_initial.full() if hasattr(rho_sys_initial, 'full') else rho_sys_initial.copy()

    projectors = np.array([P.full() if hasattr(P, 'full') else P for P in projectors], dtype=complex)
    
    # ------------------
    # Evolution Operator
    # ------------------
    V_np = V.full() if hasattr(V, 'full') else V
    U_diag_np = U_diag.full() if hasattr(U_diag, 'full') else U_diag
    U_step = V_np @ U_diag_np @ V_np.conj().T
    U_step_dag = U_step.conj().T

    # ----------
    # Dimensions
    # ----------
    dim_sys = rho_sys.shape[0]
    dim_anc = rho_anc.shape[0]

    # --------------------------
    # Call JIT-compiled function
    # --------------------------
    pops_complete = _compute_trace_ancilla_core_general(rho_sys, rho_anc, U_step, U_step_dag, 
        projectors, times, dim_sys, dim_anc, N_site )
    
    return pops_complete

#### Bloch Sphere

In [ ]:
@njit(cache=True)
def compute_Bloch_Sphere(psi):
    """
    Function to extract the expectation value of the Bloch's Sphere components <sigmax>, <sigmay>, <sigmaz> associated to the 2x2 space of only excited states,
    with base |10> (exc. on site 1, -z) & |01> (exc. on site 2, +z) 

    Parameters: -psi : nparray, wf at time t of the complete systems (wf site1 otimes wf site2)

    Returns: - r_x_step, float expectation value of x component <sigmax>
             - r_y_step, float expectation value of y component <sigmay>
             - r_z_step, float expectation value of z component <sigmaz>
    """
    # Flatten wave function if needed
    if psi.ndim > 1:
        psi = psi.flatten()
    
    # wf element
    c_01 = psi[1] ; c_01_conj = np.conj(c_01) # site 2 
    c_10 = psi[2] ; c_10_conj = np.conj(c_10) # site 1

    # Blochs components
    r_x_step = 2 * np.real(c_10 * c_01_conj)
    
    r_y_step = -2 * np.imag(c_10 * c_01_conj)

    r_z_step = np.abs(c_01)**2 - np.abs(c_10)**2

    return r_x_step, r_y_step, r_z_step

#### Single trajectory

In [ ]:
@njit(parallel=True, cache=True)
def _trajectory_core(N_traj, n_times, N_site, dt, c_CM, U_site, psi_sys_initial, 
                     projectors, Sz_ops, is_QJ, seeds, n_samples=5):
    """
    Core evolution loop - saves only n_samples complete trajectories, accumulates others in sum arrays
    
    Parameters:
        n_samples: number of complete trajectories to save (default=5)
    
    Returns:
        - pop_samples: (N_site, n_times, n_samples) - complete trajectories
        - pop_sum: (N_site, n_times) - sum of all trajectories for averaging
        - count: (N_traj,) - collision counts per trajectory
        - bloch_samples: (n_times, n_samples, 3) - Bloch coordinates for samples
        - bloch_sum: (n_times, 3) - sum of Bloch coordinates for averaging
    """
    count = np.zeros(N_traj, dtype=np.int64)
    
    # Save only n_samples complete trajectories
    pop_samples = np.zeros((N_site, n_times, n_samples), dtype=np.float64)
    bloch_samples = np.zeros((n_times, n_samples, 3), dtype=np.float64)  # [x, y, z]
    
    # Accumulate ALL trajectories for averaging
    pop_sum = np.zeros((N_site, n_times), dtype=np.float64)
    bloch_sum = np.zeros((n_times, 3), dtype=np.float64)  # [x, y, z]
    
    if is_QJ:
        jump_probabilities = np.sin(c_CM * dt)**2
    else:
        jump_probability = 0.5
        cos_vals = np.cos(c_CM * dt)
        sin_vals = np.sin(c_CM * dt)
    
    for traj in prange(N_traj):
        # set same random number sequency for a single trajectory
        np.random.seed(seeds[traj])
        
        psi = psi_sys_initial.copy()
        
        # Determine if this trajectory should be saved completely
        save_complete = (traj < n_samples)
        
        # =============
        # Initial state
        # =============
        for site in range(N_site):
            pop_val = np.real(np.vdot(psi, projectors[site] @ psi))
            
            if save_complete:
                pop_samples[site, 0, traj] = pop_val
            
            pop_sum[site, 0] += pop_val
        
        rx, ry, rz = compute_Bloch_Sphere(psi)
        
        if save_complete:
            bloch_samples[0, traj, 0] = rx
            bloch_samples[0, traj, 1] = ry
            bloch_samples[0, traj, 2] = rz
        
        bloch_sum[0, 0] += rx
        bloch_sum[0, 1] += ry
        bloch_sum[0, 2] += rz
        
        # ==============
        # Time evolution
        # ==============
        for step in range(1, n_times):
            psi = U_site @ psi
            
            # Collisions
            if is_QJ:
                for site_index in range(N_site):
                    rn_site = np.random.rand()
                    if rn_site < jump_probabilities[site_index]:
                        psi = Sz_ops[site_index] @ psi
                        count[traj] += 1
            else:
                for site_index in range(N_site):
                    Sz_psi = Sz_ops[site_index] @ psi
                    rn_site = np.random.rand()
                    
                    if rn_site < jump_probability:
                        psi = cos_vals[site_index] * psi - 1j * sin_vals[site_index] * Sz_psi
                    else:
                        psi = cos_vals[site_index] * psi + 1j * sin_vals[site_index] * Sz_psi
                        count[traj] += 1
            
            # Normalization
            norm = np.sqrt(np.real(np.vdot(psi, psi)))
            psi = psi / norm
            
            # Calculate populations
            for site in range(N_site):
                pop_val = np.real(np.vdot(psi, projectors[site] @ psi))
                
                if save_complete:
                    pop_samples[site, step, traj] = pop_val
                
                pop_sum[site, step] += pop_val
            
            # Calculate Bloch coordinates
            rx, ry, rz = compute_Bloch_Sphere(psi)
            
            if save_complete:
                bloch_samples[step, traj, 0] = rx
                bloch_samples[step, traj, 1] = ry
                bloch_samples[step, traj, 2] = rz
            
            bloch_sum[step, 0] += rx
            bloch_sum[step, 1] += ry
            bloch_sum[step, 2] += rz
    
    return pop_samples, pop_sum, count, bloch_samples, bloch_sum


def compute_trajectory_wf(dt, c_CM, N_traj, N_site, times, projectors, psi_sys_initial, U_site, 
                          method=None, n_samples=5):
    """
    Compute quantum trajectory evolution with optimized memory usage.
    Saves only n_samples complete trajectories, accumulates others for averaging.
    
    Parameters:
        - dt : float, Time Step
        - c_CM : array, Collisional model Coefficients
        - N_traj : int, Number of Trajectories
        - N_site : int, Number of Sites
        - times : array, Time array
        - projectors : Np array, Projection Operators
        - psi_sys_initial : Np array, Initial Wave Function
        - U_site : Np array, Time Evolution Operator
        - method : str, 'QJ' or 'Diff'
        - n_samples : int, Number of complete trajectories to save (default=3)
        
    Returns:
        - pop_traj_samples : np array (N_site, len(times), n_samples), Complete trajectories
        - average_pop_traj : np array (N_site, len(times)), Average over all trajectories
        - count : np array (N_traj,), Collision counts per trajectory
        - avg_count : float, Average collision count
        - r_x_samples, r_y_samples, r_z_samples : np arrays (len(times), n_samples), Bloch samples
        - avg_r_x, avg_r_y, avg_r_z : np arrays (len(times),), Average Bloch coordinates
    """
    if method is None:
        method = INTERACTION_LIMIT
    
    # Convert to NumPy
    U_site = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=np.complex128)
    psi_sys_initial = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=np.complex128).copy()
    
    U_site = np.ascontiguousarray(U_site)
    psi_sys_initial = np.ascontiguousarray(psi_sys_initial.flatten())
    
    n_times = len(times)
    
    # Prepare projectors
    projectors_np = np.zeros((N_site, *U_site.shape), dtype=np.complex128)
    for i, proj in enumerate(projectors):
        projectors_np[i] = proj.full() if hasattr(proj, 'full') else np.array(proj, dtype=np.complex128)
        projectors_np[i] = np.ascontiguousarray(projectors_np[i])
    
    # Prepare Sz operators
    Sz_ops = np.zeros((N_site, *U_site.shape), dtype=np.complex128)
    dim = 2**N_site
    
    for idx in range(N_site):
        Sz = np.zeros((dim, dim), dtype=np.complex128)
        for i in range(dim):
            if (i >> (N_site - 1 - idx)) & 1:
                Sz[i, i] = -1.0
            else:
                Sz[i, i] = 1.0
        Sz_ops[idx] = Sz
    
    c_CM = np.array(c_CM, dtype=np.float64)
    is_QJ = (method == 'QJ')

    # Setting seeds
    rng_seeds = np.random.RandomState(42) 
    seeds = rng_seeds.randint(0, 2**30, size=N_traj)
    
    # Call JIT-compiled function
    pop_samples, pop_sum, count, bloch_samples, bloch_sum = _trajectory_core(
        N_traj, n_times, N_site, dt, c_CM, U_site, psi_sys_initial,
        projectors_np, Sz_ops, is_QJ, seeds, n_samples)
    
    # Calculate averages
    average_pop_traj = pop_sum / N_traj
    avg_count = np.mean(count)
    
    avg_r_x = bloch_sum[:, 0] / N_traj
    avg_r_y = bloch_sum[:, 1] / N_traj
    avg_r_z = bloch_sum[:, 2] / N_traj
    
    # Extract Bloch coordinate samples
    r_x_samples = bloch_samples[:, :, 0].T  # (n_samples, n_times)
    r_y_samples = bloch_samples[:, :, 1].T
    r_z_samples = bloch_samples[:, :, 2].T
    
    return (pop_samples, average_pop_traj, count, avg_count, 
            r_x_samples, r_y_samples, r_z_samples, 
            avg_r_x, avg_r_y, avg_r_z)

## Main Loop for varying $ dt $ and $ N_{traj} $

In [ ]:
# -------------------
# System's Parameters
# -------------------
np.random.seed(1) # always use the same seed 
N_site = 2            # Number of sites
V_array = [1.0]    # Hopping Potential : V12, V13, ... V1N_site, V23, ..., V2N_site, V34...V3N_site
E = 1.5 + np.random.randn(N_site)*0.1     #random inizialization of the system energies

# -------------------------
# Time Evolution Parameters
# -------------------------
dt_list = [0.01]     # Time step
tf = 50.0    # Final Time
steps_list = [ int(tf / dt_list[i]) for i in range (len(dt_list)) ]
times_list = [ np.linspace(0, tf, int(steps_list[i])) for i in range(len(dt_list))]

N_traj_list = [100, 1000, 10000]

# -------------------
# Dephasing Parameter
# -------------------
g_deph = 0.1       # Gamma rate

# --------------
# Lindblad Rates
# --------------
gamma_k = [g_deph, g_deph]

# ----------------------------------------------------------
# Scaling for the collsional algorithm c = sqrt(gamma / 4dt)
# ----------------------------------------------------------
c_CM_list = np.array([[np.sqrt(g_deph / (4 * dt_list[j])) for j in range(len(dt_list))] for _ in range(N_site)])  # same Coupling for the 2 sites

#array_to_latex(c_CM)
#array_to_latex(E)

In [ ]:
# ========================================
# Initial wave function and density matrix
# ========================================

# ------
# System
# ------
psi_sys_initial = tensor(basis(2, 0), basis(2, 1)) # I set the population only in site 2
rho_sys_initial = (ket2dm(psi_sys_initial)).full()    
# ----------
# Ancilla QJ
# ----------
psi_anc_single_QJ = basis(2, 0)
rho_anc_single_QJ = ket2dm(psi_anc_single_QJ ) # Pure state |0><0| for a singe ancilla 
rho_anc_all_QJ = (tensor([rho_anc_single_QJ for _ in range(N_site)])).full() #for N ancilla

# ------------
# Ancilla Diff
# ------------
psi_anc_single_Diff_0 = (basis(2, 0)).full() # Pure state |0><0| for a singe ancilla
psi_anc_single_Diff_1 = (basis(2, 1)).full() # Pure state |1><1| for a singe ancilla
rho_anc_single_Diff = qeye(2) / 2 # Completely Mixed  1/2 (Identity) 
rho_anc_all_Diff = (tensor([rho_anc_single_Diff for _ in range(N_site)])).full() #for N ancilla

# ----------------------------
# Projectors on System's sites
# ----------------------------
P0 = (np.eye(2, dtype=complex) + sz) / 2 # projector on |0>
P1 = (np.eye(2, dtype=complex) - sz) / 2 # projector su |1>

P_00 = np.kron(P0, P0) # |00><00|
P_01 = np.kron(P0, P1) # |01><01|
P_10 = np.kron(P1, P0) # |10><10|
P_11 = np.kron(P1, P1) # |11><11|

projectors = np.array([P_10, P_01], dtype=complex) # for only excited states

# ----------------------
# Lindblad Jump Operator
# ----------------------
L_1 = P_10  # projector on |10><10|
L_2 = P_01  # projector on |01><01|
L_k = [L_1, L_2]

### QJ or Diff Limit Selection

In [ ]:
                                                   # +++++++++++++++++++++ 
INTERACTION_LIMIT = 'Diff'  # 'QJ' or 'Diff'       # + Change limit here + 
                                                   # +++++++++++++++++++++ 

### Calculation

In [ ]:
import time   # to evaluate the duration of every calculation

# Dictionary for results
results = {}

print("Starting computation for different dt and N_traj")

# Loop over dt
for dt_idx, dt in enumerate(dt_list):

    print("=" * 40)
    print(f"Processing dt = {dt:.4f} ({dt_idx+1}/{len(dt_list)})")
    

    # Initialize dictionary for this dt
    results[dt] = {}
    
    # Extract parameters for this dt
    times = times_list[dt_idx]
    steps = steps_list[dt_idx]
    c_CM = c_CM_list[:, dt_idx]  # Column corresponding to this dt
    
    # =====================================
    # Recalculate Hamiltonian and Operators
    # =====================================
    print(f"Recalculating Hamiltonians for dt = {dt:.4f}")       # because c_CM depends on dt
    
    # Hamiltonian & U for Density Matrix (with ancillas)
    H_site, H_coll, H_tot = hamiltonian_N_ancillas(N_site, E, V_array, c_CM)
    
    U_tot, U_diag, w, V = evolution_operator(H_tot, dt, method='diagonalization', hermitian=True)
    U_diag_dag = U_diag.conj().T; V_dag = V.conj().T
    
    # Hamiltonian & U for Wave Function (system only)
    H_system = system_Hamiltonian(N_site, E, V_array, mode="complete")
    
    U_site, U_diag_site, w_site, V_site = evolution_operator(H_system, dt, method='diagonalization', hermitian=True)
        
    # =========
    # Lindblad
    # =========
    print("Computing Lindblad")
    start_time = time.time()
    
    rho_list_lindblad, V_lindblad, W_lindblad = Lindblad_evo(rho_sys_initial, H_system, gamma_k, L_k, times, method="diagonal", vectorized=False)
    
    lindblad_time = time.time() - start_time
    
    print(f"Completed in {lindblad_time:.2f}s")
    
    # ==============
    # Trace Ancilla
    # ==============
    print(f"Computing Trace Ancilla")
    start_time = time.time()
    
    pops_trace = compute_trace_ancilla(rho_sys_initial, U_diag, V, times, projectors, N_site)   
    
    trace_time = time.time() - start_time
    
    print(f"Completed in {trace_time:.2f}s")

    # ========================================
    # Trajectory Isolated (without collisions)
    # ========================================
    print(f"Computing Trajectory Isolated")
    start_time = time.time()
        
    pop_traj_isolated = compute_trajectory_wf_isolated(N_site, times, projectors, psi_sys_initial, U_site)
        
    isolated_time = time.time() - start_time
    
    print(f"Completed in {isolated_time:.2f}s")
    
    # ================
    # Loop over N_traj
    # ================
    for N_traj in N_traj_list:

        print("-" * 40)
        print(f"N_traj = {N_traj}")
        
        # Initialize dictionary for this N_traj
        results[dt][N_traj] = {}
        
        # ===================================
        # Trajectory for WF (with collisions)
        # ===================================
        print(f"Computing Trajectory WF")
        start_time = time.time()
        
        # pop_traj saves only 5 complete trajectories
        pop_traj_samples, avg_pop_traj, count, avg_count, r_x_samples, r_y_samples, r_z_samples, avg_r_x, avg_r_y, avg_r_z = compute_trajectory_wf(dt, c_CM, 
                                                                                    N_traj, N_site, times, projectors, psi_sys_initial, U_site, n_samples=5)
                                                       
        traj_time = time.time() - start_time
        
        print(f"Completed in {traj_time:.2f}s")
        
        # ==================================
        # Save the Results in the Dictionary
        # ==================================
        
        results[dt][N_traj] = {
            'parameters': {
                'dt': dt,
                'N_traj': N_traj,
                'times': times,
                'steps': steps,
                'c_CM': c_CM.copy()  
            },
            
            # Trace Ancilla 
            'anc_trace': pops_trace,

             # Trajectory Isolated
            'trajectory_isolated': pop_traj_isolated,
            
            # Lindblad 
            'lindblad': {
                'rho_list': rho_list_lindblad,
                'V': V_lindblad,
                'W': W_lindblad
            },

            # Trajectory WF
            'trajectory_wf': {
                'pop_traj_samples': pop_traj_samples,  # (N_site, len(times), 3) 
                'average_pop': avg_pop_traj,           # (N_site, len(times))
                'count': count,                         # (N_traj,)
                'avg_count': avg_count,
                'bloch': {
                    'avg_x': avg_r_x,                   # (len(times),)
                    'avg_y': avg_r_y,
                    'avg_z': avg_r_z,
                    'samples_x': r_x_samples,           # (3, len(times)) 
                    'samples_y': r_y_samples,           # (3, len(times))
                    'samples_z': r_z_samples            # (3, len(times))
                } 
            }        
        }
        
        print(f"Results saved in results[{dt}][{N_traj}]")

print("\n" + "=" * 40)
print("COMPUTATION COMPLETED!")
print(f" Results saved for:")
print(f"   - {len(dt_list)} dt values: {dt_list}")
print(f"   - {len(N_traj_list)} N_traj values: {N_traj_list}")
print("=" * 40)


In [ ]:
import pickle
import os

# ----------------------
# Creating new directory
# ----------------------
results_dir = "../Results/Data"

os.makedirs(results_dir, exist_ok=True)

# -----------------
# Saving Dictionary
# -----------------
filename = os.path.join(results_dir, f"results_{INTERACTION_LIMIT}.pkl") 

with open(filename, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to: {filename}")
print(f"File size: {os.path.getsize(filename) / (1024**2):.2f} MB")